# SBERT Representation Analysis

This notebook evaluates the SBERT embeddings based on the "April 7th" requirements:
1. **Semantic Robustness**: "lonely" vs "isolated" Overlap@10.
2. **Truncation Impact**: Cosine shift between first 256 and next 256 tokens.
3. **Lexical Bias**: Keyword overlap vs semantic context.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Add project root to path
sys.path.append(os.path.abspath(".."))

from src.config import PROCESSED_DIR, FMA_METADATA_DIR
from src.embeddings.sbert import SentenceBERTEmbeddingGenerator
from src.indexing.faiss_index import FaissIndex
from src.metadata import load_tracks

### Load Data and Models

In [ ]:
generator = SentenceBERTEmbeddingGenerator()
index = FaissIndex.load(PROCESSED_DIR / "sbert_faiss.index")
df_meta = pd.read_csv(PROCESSED_DIR / "metadata_texts.csv", index_col=0)
tracks = load_tracks(FMA_METADATA_DIR)
print(f"Loaded {len(df_meta)} metadata strings and FAISS index.")

### A. Semantic Robustness: "lonely" vs "isolated"

In [ ]:
query1 = "lonely"
query2 = "isolated"

emb1 = generator.embed_text([query1])
emb2 = generator.embed_text([query2])

k = 10
results1 = index.query(emb1, k=k)
results2 = index.query(emb2, k=k)

ids1 = set([res[0] for res in results1])
ids2 = set([res[0] for res in results2])

overlap = len(ids1 & ids2)
print(f"Overlap@10 between '{query1}' and '{query2}': {overlap / k * 100:.1f}%")
print(f"Common tracks: {ids1 & ids2}")

### B. Truncation Impact

In [ ]:
# Find a track with a long description if possible, or simulate
long_track_id = df_meta.index[0]
long_text = tracks.loc[long_track_id, ('track', 'information')]
if pd.isna(long_text) or len(str(long_text)) < 500:
    # Sample a track that might have more info or just use a placeholder
    long_text = " ".join(["This is a repeated description about music."] * 50)

tokens = generator.model.tokenizer.tokenize(str(long_text))
first_part = generator.model.tokenizer.convert_tokens_to_string(tokens[:256])
second_part = generator.model.tokenizer.convert_tokens_to_string(tokens[256:512])

if second_part:
    emb_first = generator.embed_text([first_part])
    emb_second = generator.embed_text([second_part])
    cos_sim = cosine_similarity(emb_first, emb_second)[0][0]
    print(f"Cosine similarity between first 256 and next 256 tokens: {cos_sim:.4f}")
else:
    print("Text too short for truncation analysis.")

### C. Lexical Bias

In [ ]:
# Find tracks with "Blue" in title but NOT genre "Blues"
blue_mask = tracks[('track', 'title')].str.contains("Blue", case=False, na=False)
not_blues_mask = tracks[('track', 'genre_top')] != "Blues"
lexical_bias_candidates = tracks[blue_mask & not_blues_mask]

if not lexical_bias_candidates.empty:
    example = lexical_bias_candidates.iloc[0]
    print(f"Candidate for Lexical Bias: '{example[('track', 'title')]}' (Genre: {example[('track', 'genre_top')]})")
    
    query = "sad blues song"
    q_emb = generator.embed_text([query])
    results = index.query(q_emb, k=10)
    
    print(f"\nTop 10 for query '{query}':")
    for tid, score in results:
        t_row = tracks.loc[tid]
        print(f"- {t_row[('track', 'title')]} by {t_row[('artist', 'name')]} (Genre: {t_row[('track', 'genre_top')]}) [Score: {score:.4f}]")
else:
    print("No suitable candidates found for Lexical Bias analysis.")